In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existi

In [3]:
import os
import dagshub
import mlflow
from mlflow.tracking import MlflowClient
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_USERNAME"] = "mkoko22"
os.environ["MLFLOW_TRACKING_PASSWORD"] = secret_value_0

dagshub.auth.add_app_token(secret_value_0)
dagshub.init(repo_owner='mkoko22', repo_name='Titanic-Kaggle-Competition-', mlflow=True)

Accessing as mkoko22

Initialized MLflow to track repo "mkoko22/Titanic-Kaggle-Competition-"

Repository mkoko22/Titanic-Kaggle-Competition- initialized!

In [4]:
RUN_ID = 'a539c627df854d57a0141df2e60e4e87'

In [5]:
client = MlflowClient()
run_data = client.get_run(RUN_ID).data

median_age = float(run_data.params.get("imputation_median_age", 28.0))
median_fare = float(run_data.params.get("imputation_median_fare", 14.4542))
mode_embarked = run_data.params.get("imputation_mode_embarked", "S")

print(f"Loaded Imputation Values -> Age: {median_age}, Fare: {median_fare}, Embarked: {mode_embarked}")

def preprocess_test_data(df):
    data = df.copy()
    if 'Cabin' in data.columns:
        data.drop('Cabin', axis=1, inplace=True)
        
    data['Age'] = data['Age'].fillna(median_age)
    data['Fare'] = data['Fare'].fillna(median_fare)
    data['Embarked'] = data['Embarked'].fillna(mode_embarked)
    
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
    
    columns_to_drop = ['Name', 'Ticket', 'PassengerId', 'SibSp', 'Parch']
    data = data.drop(columns_to_drop, axis=1, errors='ignore')
    
    if 'Embarked' in data.columns:
        data = pd.get_dummies(data, columns=['Embarked'], drop_first=True)
        
    expected_columns = [
        'Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 
        'Embarked_Q', 'Embarked_S'
    ]
    
    for col in expected_columns:
        if col not in data.columns:
            data[col] = 0.0 
            
    data = data[expected_columns].astype(float)
    
    return data

Loaded Imputation Values -> Age: 28.0, Fare: 14.4542, Embarked: S


In [6]:
test_raw = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
passenger_ids = test_raw['PassengerId']

X_test = preprocess_test_data(test_raw)
print("Preprocessing complete. Data shape:", X_test.shape)
display(X_test.head())

print("\nLoading Registered Model 'DecisionTree' (Version 1)...")
model_uri = "models:/DecisionTree/1"

try:
    model = mlflow.sklearn.load_model(model_uri)
    print("Model Loaded into memory!")
    
    print("Generating predictions...")
    predictions = model.predict(X_test)

    submission = pd.DataFrame({
        'PassengerId': passenger_ids,
        'Survived': predictions.astype(int)
    })

    submission.to_csv('submission.csv', index=False)
    print("\nPredictions complete! Saved to 'submission.csv'")
    display(submission.head())

except Exception as e:
    print(f"Error: {e}")

Preprocessing complete. Data shape: (418, 7)


,Pclass,Sex,Age,Fare,FamilySize,Embarked_Q,Embarked_S
0,3.0,0.0,34.5,7.8292,1.0,1.0,0.0
1,3.0,1.0,47.0,7.0000,2.0,0.0,1.0
2,2.0,0.0,62.0,9.6875,1.0,1.0,0.0
3,3.0,0.0,27.0,8.6625,1.0,0.0,1.0
4,3.0,1.0,22.0,12.2875,3.0,0.0,1.0



Loading Registered Model 'DecisionTree' (Version 1)...


Model Loaded into memory!
Generating predictions...

Predictions complete! Saved to 'submission.csv'


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
